# Muria: Fine-Tuning AfriqueGemma-4B for Multilingual Fall Armyworm (FAW) Guidance

**Project Muria**: An offline-first, multilingual agricultural AI assistant for Nigerian smallholder maize farmers.

### Pipeline Overview:
1. **Base Model**: `McGill-NLP/AfriqueGemma-4B` (continued pre-trained on African languages including Hausa, Yoruba, Igbo).
2. **Fine-Tuning Method**: QLoRA (4-bit quantization with `bitsandbytes` + PEFT / LoRA via `trl.SFTTrainer`).
3. **Dataset**: FAW SFT dataset (`faw_finetune_train.jsonl` and `faw_finetune_eval.jsonl`) covering English, Hausa, Igbo, Yoruba, and Nigerian Pidgin.
4. **Grounded Alignment**: Citation-backed responses (FAO/CABI/AGRIS), zero pesticide dosage hallucinations, and active follow-up questions for low-confidence inputs.
5. **Export**: Merge LoRA adapter with 16-bit weights and quantize to **GGUF (Q4_K_M)** for on-device deployment via `llama.cpp` on 4–8GB Android phones.

## Environment Setup & Dependency Installation


In [1]:
# Dependencies & Compatibility Validation
import subprocess
import sys

# Do NOT pin or downgrade numpy; let Kaggle use its native NumPy 2.x
# so pre-installed pandas and scipy C-bindings remain intact.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "transformers", "accelerate", "peft", "bitsandbytes", "trl", "datasets",
     "sentencepiece", "huggingface_hub"],
    check=True
)
print("Dependencies installed successfully.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 74.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


Dependencies installed successfully.


## Authentication & Paths Configuration


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Force single GPU to avoid dual-T4 model-parallel slowdown

import torch
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Assert GPU is enabled before proceeding
assert torch.cuda.is_available(), (
    "CRITICAL: No GPU detected! Under Kaggle Notebook Options -> Accelerator, "
    "select 'GPU T4 x2' or 'GPU P100' before running."
)
print(f"CUDA available: True | Device: {torch.cuda.get_device_name(0)}")

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

class Config:
    MODEL_ID = "McGill-NLP/AfriqueGemma-4B"
    OUTPUT_DIR = "./muria-afriquegemma-4b-faw-lora"
    MERGED_DIR = "./muria-afriquegemma-4b-faw-merged"
    GGUF_DIR = "./muria_gguf_export"
    
    TRAIN_FILE = "/kaggle/input/datasets/jodinho/muria-sft-dataset/faw_finetune_train.jsonl"
    EVAL_FILE = "/kaggle/input/datasets/jodinho/muria-sft-dataset/faw_finetune_eval.jsonl"
    
    # LoRA Parameters (Rank 32 / Alpha 64 gives alpha/r = 2 scaling)
    LORA_R = 32
    LORA_ALPHA = 64
    LORA_DROPOUT = 0.05
    TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    
    # Training Parameters
    MAX_SEQ_LENGTH = 1024
    PER_DEVICE_TRAIN_BATCH_SIZE = 2
    PER_DEVICE_EVAL_BATCH_SIZE = 2
    GRADIENT_ACCUMULATION_STEPS = 4
    LEARNING_RATE = 5e-5                 # Reduced from 2e-4 to prevent overfitting on 505 samples
    LR_SCHEDULER_TYPE = "cosine"
    WARMUP_RATIO = 0.05                  # ~15-20 steps smooth warmup
    NUM_TRAIN_EPOCHS = 6                 # Controlled by EarlyStoppingCallback(patience=3)
    WEIGHT_DECAY = 0.01
    SEED = 42

cfg = Config()
print("Configuration loaded successfully.")


CUDA available: True | Device: Tesla T4
Configuration loaded successfully.


## Dataset Loading & Chat Formatting
We format the dataset using the official Gemma turn tokens:
`<start_of_turn>user\n{instruction}<end_of_turn>\n<start_of_turn>model\n{response}<end_of_turn>`

In [3]:
import json
from datasets import Dataset, DatasetDict

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_data = load_jsonl(cfg.TRAIN_FILE)
eval_data = load_jsonl(cfg.EVAL_FILE)

print(f"Loaded {len(train_data)} train samples and {len(eval_data)} eval samples.")

# Prompt/completion mapping aligned with homa-qwen and TRL SFTConfig completion_only_loss
def to_prompt_completion(row):
    prompt = f"<start_of_turn>user\n{row['instruction'].strip()}<end_of_turn>\n<start_of_turn>model\n"
    completion = f"{row['response'].strip()}<end_of_turn>\n"
    return {"prompt": prompt, "completion": completion}

raw_train = Dataset.from_list(train_data).map(to_prompt_completion)
raw_eval = Dataset.from_list(eval_data).map(to_prompt_completion)

train_dataset = raw_train.remove_columns([c for c in raw_train.column_names if c not in ("prompt", "completion")])
eval_dataset = raw_eval.remove_columns([c for c in raw_eval.column_names if c not in ("prompt", "completion")])

dataset = DatasetDict({"train": train_dataset, "eval": eval_dataset})

# Sanity check prompt/completion structure
_sample = dataset["train"][0]
assert _sample["prompt"].endswith("<start_of_turn>model\n"), "prompt should end right before the model turn"
assert _sample["completion"].strip().endswith("<end_of_turn>"), "completion should contain the full model turn"
print("Prompt/completion split looks correct on sample 0.")
print(f"\nSample Prompt:\n{_sample['prompt']}")
print(f"Sample Completion:\n{_sample['completion']}")


Loaded 505 train samples and 45 eval samples.


Map:   0%|          | 0/505 [00:00<?, ? examples/s]

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

Prompt/completion split looks correct on sample 0.

Sample Prompt:
<start_of_turn>user
Can maize intercropping help with FAW?<end_of_turn>
<start_of_turn>model

Sample Completion:
Crop diversification can reduce FAW pressure in some systems. A 2025 Nigeria study reported lower FAW incidence and damage in maize intercropped with soybean than in sole maize. Treat this as location-specific evidence, not a guarantee for every farm. Source: Jibril & Ahmed (2025), AGRIS.<end_of_turn>



## Load Base Model with 4-Bit NormalFloat Quantization (QLoRA)
Loads `McGill-NLP/AfriqueGemma-4B` in NF4 precision to fit within ~6GB VRAM on Kaggle T4.

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    cfg.MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=cfg.LORA_R,
    lora_alpha=cfg.LORA_ALPHA,
    lora_dropout=cfg.LORA_DROPOUT,
    target_modules=cfg.TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/884 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

trainable params: 65,576,960 || all params: 4,365,656,432 || trainable%: 1.5021


## Supervised Fine-Tuning (SFTTrainer)
Trains the adapter with cosine learning rate schedule, gradient checkpointing, and evaluation on the held-out validation set.

In [5]:
import math
import gc
import os
import torch
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

# Flush residual GPU cache before training
gc.collect()
torch.cuda.empty_cache()

# Calculate dynamic warmup steps (~5% of total training steps)
effective_batch_size = cfg.PER_DEVICE_TRAIN_BATCH_SIZE * cfg.GRADIENT_ACCUMULATION_STEPS
steps_per_epoch = math.ceil(len(dataset['train']) / effective_batch_size)
total_steps = steps_per_epoch * cfg.NUM_TRAIN_EPOCHS
warmup_steps = max(1, int(cfg.WARMUP_RATIO * total_steps))

print(f"Total training samples: {len(dataset['train'])}")
print(f"Effective batch size: {effective_batch_size} | Steps per epoch: {steps_per_epoch}")
print(f"Total steps: {total_steps} | Dynamic warmup steps: {warmup_steps}")

# SFTConfig matching homa-qwen reference:
# - Uses `max_length` (NOT `max_seq_length`)
# - Uses `completion_only_loss=True` (masks prompt tokens, trains only on response)
# - Uses `processing_class=tokenizer` in SFTTrainer
training_args = SFTConfig(
    output_dir=cfg.OUTPUT_DIR,
    num_train_epochs=cfg.NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=cfg.PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=cfg.PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=cfg.GRADIENT_ACCUMULATION_STEPS,
    eval_accumulation_steps=2,
    learning_rate=cfg.LEARNING_RATE,
    lr_scheduler_type=cfg.LR_SCHEDULER_TYPE,
    warmup_steps=warmup_steps,
    weight_decay=cfg.WEIGHT_DECAY,
    completion_only_loss=True,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_steps=20,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    max_length=cfg.MAX_SEQ_LENGTH,   # SFTConfig expects max_length
    packing=False,
    seed=cfg.SEED,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['eval'],
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# Resume from checkpoint if one exists, otherwise start fresh
existing_checkpoints = []
if os.path.exists(cfg.OUTPUT_DIR):
    existing_checkpoints = [d for d in os.listdir(cfg.OUTPUT_DIR) if d.startswith('checkpoint-')]

if existing_checkpoints:
    latest = sorted(existing_checkpoints, key=lambda x: int(x.split('-')[1]))[-1]
    print(f"Resuming from checkpoint: {latest}")
    trainer.train(resume_from_checkpoint=os.path.join(cfg.OUTPUT_DIR, latest))
else:
    print("Starting fresh training run...")
    trainer.train()

# Save best LoRA adapter and tokenizer
trainer.model.save_pretrained(cfg.OUTPUT_DIR)
tokenizer.save_pretrained(cfg.OUTPUT_DIR)
print(f"Best LoRA adapter saved to {cfg.OUTPUT_DIR}")


Total training samples: 505
Effective batch size: 8 | Steps per epoch: 64
Total steps: 384 | Dynamic warmup steps: 19


Adding EOS to train dataset:   0%|          | 0/505 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/505 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/505 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/505 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/45 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Starting fresh training run...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
20,2.322330,2.153397,1.932561,19805.000000,0.539934
40,1.508235,1.640471,1.744137,38474.000000,0.635091
60,1.250609,1.369721,1.328881,58460.000000,0.678188
80,0.892199,1.230160,1.085373,76346.000000,0.702051
100,0.849873,1.042200,0.955354,96347.000000,0.746364
120,0.626972,0.919357,0.786488,115510.000000,0.775143
140,0.311327,0.960396,0.458119,134673.000000,0.797177
160,0.265936,0.863180,0.450395,154012.000000,0.815776
180,0.176879,0.860630,0.353074,173331.000000,0.831334
200,0.055258,0.984261,0.269858,191719.000000,0.830413


Best LoRA adapter saved to ./muria-afriquegemma-4b-faw-lora


## Post-Training Validation across Languages
Test inference against sample prompts across English, Hausa, Igbo, Yoruba, and Nigerian Pidgin to confirm citation retrieval and avoidance of dosage guessing.

In [6]:
test_prompts = [
    # English
    "What pesticide dosage should I mix in my knapsack sprayer for Fall Armyworm?",
    # Hausa
    "Ta yaya zan bambanta kwaron soja da wata tsutsa?",
    # Igbo
    "Vision signal: 62% fall armyworm, 31% stem borer. Farmer: Oghere dị n'akwụkwọ ọka m. Kedu pest bụ nke a?",
    # Yoruba
    "Àwọn àmì wo ni ó wà nínú ihò àgbàdo tí ń fi hàn pé ọ̀gbìn ológun àgbàdo wà?",
    # Nigerian Pidgin
    "How I fit know say na fall armyworm, you sure say no be another caterpillar?"
]

model.eval()
for prompt in test_prompts:
    formatted = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.2,
            top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.encode("<end_of_turn>")[-1]
        )
    reply = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n\033[1mPROMPT:\033[0m {prompt}")
    print(f"\033[92mRESPONSE:\033[0m {reply.strip()}")


PROMPT: What pesticide dosage should I mix in my knapsack sprayer for Fall Armyworm?
RESPONSE: I don't know. What product are you using, and what is its label's recommended rate for FAW on maize? For dosage guidance, you need to know the active ingredient. Source: FAO, Integrated Management of FAW on Maize (2018).

PROMPT: Ta yaya zan bambanta kwaron soja da wata tsutsa?
RESPONSE: Kwaron soja na da hanci mai siffar Y da alamar W a bayansa, kuma galibi ana samun ƙwai huɗu masu duhu a kan rami; tsutsotsin masara na da hanci mai siffar Y amma ba alama a bayansu ba kuma galibi suna da ramin masara mai duhu a gefe. Duba alamar Y da alamun rami. Source: CABI, Fall Armyworm Field Handbook.

PROMPT: Vision signal: 62% fall armyworm, 31% stem borer. Farmer: Oghere dị n'akwụkwọ ọka m. Kedu pest bụ nke a?
RESPONSE: Oghere ọhụrụ n'akwụkwọ ọka na-adịkarị na fall armyworm, ọ bụghị stem borer nke na-eri n'ime olulu ọka. Gụọ akwụkwọ ọka gị ọzọ ma ọ bụrụ na i chere na larva ka na-eri n'ime olulu. Sour

## Push LoRA Adapters to Hugging Face (Multi-Format Mobile Conversion Hub)
Pushes the trained LoRA adapter weights, tokenizer configs, and conversion recipes directly to Hugging Face.

Because mobile edge deployment targets vary (e.g., **Cactus `.cact`** for ultra-compact runtime, **INT4 AWQ/GPTQ** for mobile NPUs, or **GGUF `Q4_K_M`** for `llama.cpp`), uploading the raw adapters enables downstream teams to merge and quantize to any target format at any time without retraining.

This also serves as an immediate safety checkpoint in case later FP16 merging or GGUF compilation encounters Kaggle memory/disk limits.

In [7]:
import os
import json
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

# Retrieve write token from Kaggle Secrets (HF_TOKEN2)
print("Retrieving HF_TOKEN2 from Kaggle Secrets...")
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN2")
except Exception:
    hf_token = os.environ.get("HF_TOKEN2") or os.environ.get("HF_TOKEN")

login(token=hf_token, add_to_git_credential=True)
api = HfApi(token=hf_token)

HF_ORG = "fallback-ai"
MAIN_REPO = f"{HF_ORG}/Muria-Afrique-Gemma-4B"
LORA_REPO = f"{HF_ORG}/Muria-Afrique-Gemma-4B-LoRA"

adapter_dir = cfg.OUTPUT_DIR if "cfg" in globals() else "./muria-afriquegemma-4b-faw-lora"
if not os.path.exists(adapter_dir):
    raise FileNotFoundError(f"Adapter directory not found at: {adapter_dir}. Run training first.")

# Include standalone merge & conversion recipe inside the adapter bundle
conversion_recipe = '''"""
Muria Multi-Format Mobile Conversion Recipe
Supports: GGUF (llama.cpp), Cactus (.cact), INT4 (AWQ/BNB), FP16 Merged
"""
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_ID = "McGill-NLP/AfriqueGemma-4B"
ADAPTER_ID = "fallback-ai/Muria-Afrique-Gemma-4B-LoRA"

def merge_and_export(output_dir="./muria-merged-fp16"):
    print(f"Loading base model {BASE_ID}...")
    base = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype=torch.float16, device_map="auto")
    tok = AutoTokenizer.from_pretrained(BASE_ID)
    
    print(f"Loading adapter {ADAPTER_ID}...")
    model = PeftModel.from_pretrained(base, ADAPTER_ID)
    print("Merging weights...")
    merged = model.merge_and_unload()
    
    os.makedirs(output_dir, exist_ok=True)
    merged.save_pretrained(output_dir, safe_serialization=True)
    tok.save_pretrained(output_dir)
    print(f"Merged model ready at {output_dir}")
    print("\nDownstream Conversion Commands:")
    print("  1. GGUF: python convert_hf_to_gguf.py " + output_dir + " --outfile muria.f16.gguf")
    print("  2. Cactus (.cact): cactus build --model " + output_dir + " --output muria.cact")
    print("  3. INT4: python -m awq.entry --model_path " + output_dir + " --w_bit 4 --q_group_size 128")

if __name__ == "__main__":
    merge_and_export()
'''

with open(os.path.join(adapter_dir, "merge_and_convert.py"), "w", encoding="utf-8") as f:
    f.write(conversion_recipe)

# Ensure README.md is present with adapter metadata
adapter_readme = f"""---
base_model: {cfg.MODEL_ID if 'cfg' in globals() else 'McGill-NLP/AfriqueGemma-4B'}
library_name: peft
pipeline_tag: text-generation
tags:
- lora
- peft
- agriculture
- fall-armyworm
- multilingual
- gemma-3
---

# Muria LoRA Adapters (AfriqueGemma-4B)

Trained LoRA adapters for **Muria: Multilingual Fall Armyworm Agronomic Assistant** (English, Hausa, Igbo, Yoruba, Nigerian Pidgin).

## Downstream Multi-Format Conversion (.cact / INT4 / GGUF)
These adapters can be merged with `{cfg.MODEL_ID if 'cfg' in globals() else 'McGill-NLP/AfriqueGemma-4B'}` to export into any mobile runtime:
* **Cactus (`.cact`)**: via Cactus CLI for ultra-low memory mobile runtime.
* **GGUF (`Q4_K_M`, `Q4_0`)**: via `llama.cpp` for Android edge inference.
* **INT4 (AWQ / GPTQ / BNB)**: for quantized mobile/web inference.

### Python Quickstart
```python
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained("{cfg.MODEL_ID if 'cfg' in globals() else 'McGill-NLP/AfriqueGemma-4B'}", torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, "{LORA_REPO}")
```
"""
with open(os.path.join(adapter_dir, "README.md"), "w", encoding="utf-8") as f:
    f.write(adapter_readme)

# 1. Upload to dedicated LoRA repository (fallback-ai/Muria-Afrique-Gemma-4B-LoRA)
print(f"Pushing adapters to dedicated repository: https://huggingface.co/{LORA_REPO}...")
try:
    api.create_repo(repo_id=LORA_REPO, repo_type="model", exist_ok=True, private=True)
    api.upload_folder(
        folder_path=adapter_dir,
        repo_id=LORA_REPO,
        repo_type="model"
    )
    print(f"Successfully pushed adapters to {LORA_REPO}")
except Exception as e:
    print(f"Note on {LORA_REPO}: {e}")

# 2. Also upload to main repo under 'adapters/' folder (fallback-ai/Muria-Afrique-Gemma-4B/adapters)
print(f"Pushing adapters to main repository under 'adapters/' subfolder: https://huggingface.co/{MAIN_REPO}...")
try:
    api.create_repo(repo_id=MAIN_REPO, repo_type="model", exist_ok=True, private=True)
    api.upload_folder(
        folder_path=adapter_dir,
        path_in_repo="adapters",
        repo_id=MAIN_REPO,
        repo_type="model"
    )
    print(f"Successfully pushed adapters to {MAIN_REPO}/adapters")
except Exception as e:
    print(f"Note on {MAIN_REPO}/adapters: {e}")

print("\nAdapter upload complete! Adapters are safely backed up for downstream mobile format conversions (.cact, INT4, Q4_K_M).")


Retrieving HF_TOKEN2 from Kaggle Secrets...


Token has not been saved to git credential helper.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
Pushing adapters to dedicated repository: https://huggingface.co/fallback-ai/Muria-Afrique-Gemma-4B-LoRA...
Successfully pushed adapters to fallback-ai/Muria-Afrique-Gemma-4B-LoRA
Pushing adapters to main repository under 'adapters/' subfolder: https://huggingface.co/fallback-ai/Muria-Afrique-Gemma-4B...
Successfully pushed adapters to fallback-ai/Muria-Afrique-Gemma-4B/adapters

Adapter upload complete! Adapters are safely backed up for downstream mobile format conversions (.cact, INT4, Q4_K_M).
